# 🧮 Complemento — Clase 07: hasta dónde creerle a un sistema

Este material **salió de la clase 07** para que la clase se enfoque en lo importante: plantear un
problema de gestión como sistema de ecuaciones y resolverlo.

Acá quedan dos temas más finos, para el que quiera profundizar:

1. **El número de condición**: cuánto se amplifica un error de medición en el resultado.
2. **Mínimos cuadrados** (`lstsq`): qué hacer cuando hay más condiciones que incógnitas.
3. **El business case del proceso P2P**: cómo se traduce el retrabajo en personas y en plata.

> ⚠️ **No entra en el parcial.** Es material de consulta.

---
## Parte C2 — ¿Cuánto le puedo creer al resultado?

Acá va la pregunta que separa a alguien que corre código de alguien que sabe lo que está haciendo.

Los coeficientes de la matriz **casi nunca son exactos**. "Una silla lleva 2 horas de carpintería"
sale de cronometrar unas cuantas sillas y promediar. ¿Qué pasa si ese 2 era en realidad 2,05?

Si un error chiquito en los datos produce un cambio chiquito en la respuesta, el modelo es confiable.
Si lo amplifica, el resultado **no se puede llevar a una reunión**.

El **número de condición** mide exactamente eso:

$$\frac{\Delta x}{x} \;\approx\; \operatorname{cond}(A) \cdot \frac{\Delta A}{A}$$

| cond(A) | Lectura |
|---|---|
| cerca de 1 | Excelente: el error no se amplifica |
| hasta ~100 | Aceptable |
| más de 1.000 | ⚠️ Frágil: revisá cómo se midieron los datos |

In [ ]:
print("Condición de A3:", round(np.linalg.cond(A3), 2))

# Perturbamos UN coeficiente un 2,5%: las horas de máquina del producto A pasan de 3 a 3,075
A3_perturbada = A3.astype(float).copy()
A3_perturbada[0, 0] = A3[0, 0] * 1.025

x3_perturbado = np.linalg.solve(A3_perturbada, b3)
cambio = np.abs((x3_perturbado - x3) / x3 * 100)

comparacion = pd.DataFrame({
    "Producto": ["A", "B", "C"],
    "Plan original": x3.round(1),
    "Con el dato corregido": x3_perturbado.round(1),
    "Cambio %": cambio.round(1),
})
comparacion

**Un error de medición del 2,5% mueve el plan menos del 3%.** El error no se amplifica:
nuestra matriz está sana y el resultado se puede defender.

Ahora veamos qué pasa cuando **no** lo está. Una fábrica textil produce remeras, buzos y camperas.
El problema: los tres productos consumen los recursos en proporciones **muy parecidas**, así que las
ecuaciones casi se repiten.

In [ ]:
# Consumo de recursos por producto (filas = producto, columnas = recurso)
M = np.array([[0.40, 1.20, 0.50],    # remera
              [0.90, 2.40, 1.10],    # buzo
              [1.60, 3.80, 2.00]])   # campera
capacidad = np.array([1200., 3500, 1800])

print("Determinante :", round(np.linalg.det(M), 4), "  ← casi cero")
print("Condición    :", round(np.linalg.cond(M), 1), "      ← altísima ⚠️")

In [ ]:
plan = np.linalg.solve(M.T, capacidad)

M_perturbada = M.copy()
M_perturbada[0, 0] = 0.41                 # 0,40 → 0,41 : apenas un 2,5%
plan_perturbado = np.linalg.solve(M_perturbada.T, capacidad)

textil = pd.DataFrame({
    "Producto": ["Remera", "Buzo", "Campera"],
    "Plan con 0,40": plan.round(0),
    "Plan con 0,41": plan_perturbado.round(0),
    "Cambio %": np.abs((plan_perturbado - plan) / plan * 100).round(0),
})
textil

### 🚨 Dos alarmas al precio de una

1. **El mismo cambio del 2,5% mueve el plan más del 130%.** Con condición 553, el error de medición
   se amplifica hasta volver el resultado inservible.
2. **El plan dice producir −12.000 buzos.** Matemáticamente correcto, económicamente imposible.

> 🎯 **Lo importante:** Python **no te avisó**. Devolvió números con cuatro decimales, con toda
> confianza. El determinante casi nulo y la condición altísima eran las únicas señales.
>
> Y la recomendación de gestión no es "buscá otro algoritmo": es **andá a medir mejor la planta**.
> El problema está en los datos, no en la cuenta.

---
## Parte D2 — Cuando hay más condiciones que incógnitas

Hasta acá siempre tuvimos tantas ecuaciones como incógnitas. En la realidad rara vez es así:

| Situación | Qué pasa | Qué usar |
|---|---|---|
| Igual cantidad | Puede haber solución exacta | `solve` |
| **Más ecuaciones que incógnitas** | Sobredeterminado: normalmente **no hay** solución exacta | `lstsq` |
| Menos ecuaciones que incógnitas | Infinitas soluciones: falta información | agregar un criterio (optimizar) |

Volvamos a la fábrica de electrodomésticos. Además de agotar los tres recursos, gerencia
**pide 900 unidades en total**. Ahora son 4 condiciones para 3 productos.

In [ ]:
# vstack: apilar filas. Agregamos la condición "A + B + C = 900"
A4 = np.vstack([A3, [1, 1, 1]])
b4 = np.append(b3, 900)

print("Ecuaciones:", A4.shape[0], "| Incógnitas:", A4.shape[1], " → sobredeterminado")
A4

In [ ]:
# lstsq cambia la pregunta: ya no busca cumplir todo, busca el MENOR error total
solucion, residuo, rango, _ = np.linalg.lstsq(A4, b4, rcond=None)

print("Solución de compromiso:", solucion.round(1))
print("Residuo               :", residuo.round(0))

In [ ]:
# ¿Qué tan lejos quedó de lo que se pedía?
logrado = A4 @ solucion

balance = pd.DataFrame({
    "Condición": ["Horas de máquina", "Mano de obra", "Materia prima", "Total de unidades"],
    "Pedido": b4,
    "Logrado": logrado.round(0),
    "Diferencia": (logrado - b4).round(0),
})
balance

### Cómo se lee esto

Los tres recursos se cumplen casi exactos (se pasan entre 12 y 20 unidades), pero el total queda en
**754 unidades en lugar de las 900 pedidas**.

`lstsq` no hizo magia: repartió el error donde menos dolía. Y el **residuo** es el termómetro —
es la suma de los cuadrados de lo que no se pudo cumplir:

| Residuo | Significa |
|---|---|
| Cerca de cero | Las condiciones son compatibles: hay solución exacta |
| Grande | **Las condiciones se contradicen entre sí** |

Acá el residuo es grande, y eso es **información de gestión**, no un error: con esa capacidad
instalada, pedir 900 unidades es imposible. Alguien tiene que ceder — o se amplía la planta, o se
baja la meta.

> 💡 `lstsq` es, además, el motor de la **regresión lineal**. Cuando ajustan una recta a una nube de
> puntos, están resolviendo exactamente este problema: más ecuaciones (puntos) que incógnitas
> (coeficientes), minimizando el error al cuadrado.

---
## 💼 El caso completo: del retrabajo al business case

Esta parte continúa el ejemplo del proceso P2P de la clase: una vez que sabemos cuántas facturas
toca cada etapa, se puede estimar **cuánta gente hace falta** y **cuánto cuesta el retrabajo**.

### ¿Para qué sirve el número? Para dimensionar el equipo

Cada estación tiene una productividad distinta. Con la carga real ya podemos calcular **cuánta gente
hace falta** — en la jerga, cuántos **FTE** (*Full Time Equivalent*, una persona a tiempo completo).

In [ ]:
carga["productividad"] = estaciones["productividad_mes_por_persona"].values
carga["FTE_necesarios"] = (carga["carga_real"] / carga["productividad"]).round(1)

# Con qué se compararía alguien que ignora el retrabajo
carga["FTE_si_ignoro_retrabajo"] = (10000 / carga["productividad"]).round(1)
carga["subestimación"] = (carga["FTE_necesarios"] - carga["FTE_si_ignoro_retrabajo"]).round(1)

carga[["estación", "carga_real", "FTE_necesarios", "FTE_si_ignoro_retrabajo", "subestimación"]]

In [ ]:
total_real = carga["FTE_necesarios"].sum()
total_ingenuo = carga["FTE_si_ignoro_retrabajo"].sum()

print(f"Equipo necesario (con retrabajo) : {total_real:>5.1f} FTE")
print(f"Equipo si ignoro el retrabajo    : {total_ingenuo:>5.1f} FTE")
print(f"{'-' * 45}")
print(f"Diferencia                       : {total_real - total_ingenuo:>5.1f} FTE de menos")

> 🎯 **Esto es exactamente lo que pasa en la vida real.** Alguien dimensiona el equipo con la regla de
> tres simple, contrata de menos, y después el área vive apagando incendios sin entender por qué.
> El retrabajo es invisible hasta que lo modelás.

### El business case: ¿cuánto vale arreglar el proceso?

Supongamos que se automatiza el matching de la factura contra la orden de compra, y el rechazo en
Validación baja del **18% al 8%**. ¿Cuánto se ahorra?

Volvemos a resolver el mismo sistema con el parámetro cambiado.

In [ ]:
def simular(rechazo_validacion):
    """Resuelve el sistema con otro % de rechazo y devuelve la carga y los FTE."""
    M = A.copy()
    M.loc["Recepción",  "Validación"] = rechazo_validacion        # lo que vuelve para atrás
    M.loc["Aprobación", "Validación"] = 1 - rechazo_validacion    # lo que sigue adelante
    carga_sim = np.linalg.solve(I - M.values, d)
    fte = (carga_sim / carga["productividad"].values).sum()
    return carga_sim, fte

carga_hoy, fte_hoy = simular(0.18)
carga_nueva, fte_nueva = simular(0.08)

comparacion = pd.DataFrame({
    "estación": etapas,
    "hoy (18%)": carga_hoy.round(0),
    "con mejora (8%)": carga_nueva.round(0),
    "diferencia": (carga_nueva - carga_hoy).round(0),
})
comparacion

In [ ]:
COSTO_FTE_ANUAL = 1_800_000     # costo anual de una persona, en pesos

ahorro_fte = fte_hoy - fte_nueva
ahorro_pesos = ahorro_fte * COSTO_FTE_ANUAL

print(f"Equipo hoy         : {fte_hoy:>5.1f} FTE")
print(f"Equipo con mejora  : {fte_nueva:>5.1f} FTE")
print(f"Ahorro             : {ahorro_fte:>5.1f} FTE")
print(f"\nAhorro anual: $ {ahorro_pesos:,.0f}")
print("\nSi el proyecto de automatización cuesta menos que eso, se paga solo en el primer año.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

pos = np.arange(len(etapas))
ancho = 0.35
ax.bar(pos - ancho/2, carga_hoy,   ancho, label="Hoy (18% de rechazo)",   color="#243b5e")
ax.bar(pos + ancho/2, carga_nueva, ancho, label="Con mejora (8%)",        color="#e07b39")
ax.axhline(10000, color="gray", linestyle="--", linewidth=1.5,
           label="Facturas que realmente entran (10.000)")

ax.set_xticks(pos)
ax.set_xticklabels(etapas)
ax.set_ylabel("Facturas procesadas por mes")
ax.set_title("Carga real de cada estación del proceso P2P", loc="left", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Todo lo que está **por encima de la línea gris es retrabajo**. El gráfico muestra de un vistazo dónde
duele y cuánto se recupera arreglándolo.